# 37 — Pool selection by d_cos: does the metric choose a different pool?

Notebook 36 found near-zero Spearman rank correlation (0.038) between
expression-Δ and the bounded d_cos across pool concepts — the two tracks
rank concept steerability almost independently, plausibly because
expression carries concept-specific scale that the cosine strips. Since
the bf16 pool (notebook 32) was *selected* by expression-Δ, this notebook
re-runs the identical selection algorithm ranked by **d_cos** and compares
the two pools.

Steps:
1. **Complete the d_cos table**: the 27 incumbents predate metric v3;
   their d_cos is measured by re-scoring notebook 29's stored single
   continuations with `angle_record` (baselines regenerated greedily and
   gated against the logged baseline expression values — the notebook-30
   procedure). Fresh candidates already have d_cos from notebook 32.
2. **Recompute the candidate ρ matrix** (cached concepts; no generation).
3. **Re-select** with the same rules — greedy by metric, positivity
   floor, ≤6 per lexname, ρ-dedup at 0.75, size 40 — ranked by d_cos.
4. **Compare**: overlap, adds/drops, lexname mix, cross-metric means.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F

from frames.evaluation.harness import EvaluationHarness
from frames.linalg.frame import Frame
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept

In [ ]:
MIN_LEMMAS = 11
MAX_TOKENS = 3
STEPS = 24
POOL_SIZE = 40
MAX_PER_LEXNAME = 6
RHO_DEDUP = 0.75
DCOS_FLOOR = 0.0  # positivity floor (expression selection used 0.3 in its units)

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

POOL32 = json.loads(Path("resources/32_concept_pool_bf16.json").read_text())
cand = pd.DataFrame(POOL32["pool"] + POOL32["rejected"]).set_index("synset")
POOL_EXPR = [r["synset"] for r in POOL32["pool"]]
print(
    f"{len(cand)} candidates; {cand['d_cos'].notna().sum()} with d_cos, "
    f"{int(cand['incumbent'].sum())} incumbents lacking it"
)


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    device_map="auto", id=MODEL_ID, torch_dtype=torch.bfloat16
)
harness = EvaluationHarness(
    fur, "/dev/null", min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
)
prompts = [chat(u, a) for u, a in QUESTIONS]
baseline_texts = harness.generate_baseline(prompts, max_new_tokens=STEPS + 2)
baseline_cont = {p: harness.continuation(t, p) for p, t in zip(prompts, baseline_texts)}

records = [
    json.loads(line)
    for line in Path("resources/29_llama8b_bf16.jsonl").read_text().strip().split("\n")
]
singles = [r for r in records if r["config"]["method"] == "single"]

# gate: regenerated baselines must reproduce the logged baseline expression
checked = 0
for r in singles:
    name = r["config"]["concept"]
    concept_obj = fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
    redo = harness.concept_expression(baseline_cont[r["prompt"]], concept_obj)
    assert abs(redo - r["expression"][name]["baseline"]) < 1e-2
    checked += 1
    if checked == 5:
        break
print("baseline reproduction gate passed")

incumbent_dcos = {}
for r in singles:
    name = r["config"]["concept"]
    concept_obj = fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
    a = harness.angle_record(
        r["continuation"], baseline_cont[r["prompt"]], {name: concept_obj}
    )
    incumbent_dcos.setdefault(name, []).append(a[name]["delta"]["token_cos"])
incumbent_dcos = {k: sum(v) / len(v) for k, v in incumbent_dcos.items()}
print(f"measured d_cos for {len(incumbent_dcos)} incumbents")

cand["d_cos"] = cand["d_cos"].fillna(pd.Series(incumbent_dcos))
assert cand["d_cos"].notna().all()

In [ ]:
names = sorted(cand.index)
concept = {}
for name in names:
    c = fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
    concept[name] = Concept(synset=c.synset, tensor=c.tensor.to(fur.model.device))
kmax = max(len(concept[n]) for n in names)
stack = torch.cat(
    [F.pad(concept[n].tensor, (0, kmax - len(concept[n]))) for n in names]
)
RHO = Frame.correlation(stack, stack).float().cpu()
print(f"rho matrix for {len(names)} candidates")


def select(metric: str, floor: float) -> list[str]:
    ranked = cand.sort_values(metric, ascending=False)
    pool, lex_count = [], {}
    for n, row in ranked.iterrows():
        if len(pool) == POOL_SIZE or row[metric] < floor:
            break
        if lex_count.get(row["lexname"], 0) >= MAX_PER_LEXNAME:
            continue
        i = names.index(n)
        if any(float(RHO[i, names.index(p)]) > RHO_DEDUP for p in pool):
            continue
        pool.append(n)
        lex_count[row["lexname"]] = lex_count.get(row["lexname"], 0) + 1
    return pool


POOL_DCOS = select("d_cos", DCOS_FLOOR)
# sanity: re-running the expression selection reproduces notebook 32's pool
POOL_EXPR_REDO = select("delta", 0.3)
print(
    f"expression re-selection matches notebook 32: "
    f"{set(POOL_EXPR_REDO) == set(POOL_EXPR)}"
)

In [ ]:
overlap = set(POOL_DCOS) & set(POOL_EXPR)
added = [n for n in POOL_DCOS if n not in POOL_EXPR]
dropped = [n for n in POOL_EXPR if n not in POOL_DCOS]
print(f"overlap: {len(overlap)}/40")
print(f"added by d_cos selection ({len(added)}):")
print("  " + ", ".join(n.split(".")[0] for n in added))
print(f"dropped from expression pool ({len(dropped)}):")
print("  " + ", ".join(n.split(".")[0] for n in dropped))

comparison = pd.DataFrame(
    {
        "expr_pool": {
            "mean delta": cand.loc[POOL_EXPR, "delta"].mean(),
            "mean d_cos": cand.loc[POOL_EXPR, "d_cos"].mean(),
            "lexnames": cand.loc[POOL_EXPR, "lexname"].nunique(),
            "incumbents": int(cand.loc[POOL_EXPR, "incumbent"].sum()),
        },
        "dcos_pool": {
            "mean delta": cand.loc[POOL_DCOS, "delta"].mean(),
            "mean d_cos": cand.loc[POOL_DCOS, "d_cos"].mean(),
            "lexnames": cand.loc[POOL_DCOS, "lexname"].nunique(),
            "incumbents": int(cand.loc[POOL_DCOS, "incumbent"].sum()),
        },
    }
)
print()
print(comparison.round(4).to_string())

out = {
    "config": {
        "metric": "d_cos",
        "floor": DCOS_FLOOR,
        "max_per_lexname": MAX_PER_LEXNAME,
        "rho_dedup": RHO_DEDUP,
        "incumbent_dcos_source": "re-scored notebook 29 singles",
    },
    "pool_dcos": POOL_DCOS,
    "pool_expr": POOL_EXPR,
    "added": added,
    "dropped": dropped,
    "incumbent_dcos": incumbent_dcos,
}
Path("resources/37_pool_dcos.json").write_text(json.dumps(out, indent=1))
cand.loc[POOL_DCOS].round(4).to_csv("resources/37_pool_dcos_summary.csv")
print("wrote resources/37_pool_dcos.json")

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.5))
in_both = [n in overlap for n in cand.index]
in_expr = [n in POOL_EXPR and n not in overlap for n in cand.index]
in_dcos = [n in POOL_DCOS and n not in overlap for n in cand.index]
neither = [n not in POOL_EXPR and n not in POOL_DCOS for n in cand.index]
ax.scatter(
    cand.loc[neither, "delta"],
    cand.loc[neither, "d_cos"],
    s=14,
    color="lightgray",
    label="not selected",
)
ax.scatter(
    cand.loc[in_expr, "delta"],
    cand.loc[in_expr, "d_cos"],
    s=26,
    color="tab:blue",
    label="expression pool only",
)
ax.scatter(
    cand.loc[in_dcos, "delta"],
    cand.loc[in_dcos, "d_cos"],
    s=26,
    color="tab:red",
    label="d_cos pool only",
)
ax.scatter(
    cand.loc[in_both, "delta"],
    cand.loc[in_both, "d_cos"],
    s=30,
    color="tab:green",
    label="both pools",
)
ax.set_xlabel("expression delta (model-scaled)")
ax.set_ylabel("d_cos (bounded)")
spearman = cand["delta"].corr(cand["d_cos"], method="spearman")
ax.set_title(
    f"candidate steerability under two metrics (spearman {spearman:.2f}); "
    f"pool overlap {len(overlap)}/40"
)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("resources/37_pool_dcos.png", dpi=150)
plt.show()